In [12]:
!pip -q install "transformers[torch]" datasets peft accelerate sentencepiece scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 51.3 MB/s eta 0:00:00


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import pandas as pd, numpy as np, re, html, emoji, torch
from datasets import Dataset, DatasetDict, disable_caching, ClassLabel
disable_caching()

In [15]:
CSV_PATH = "/content/merged_tornado_tweets.csv"   # 👈 adjust if needed
df = pd.read_csv(CSV_PATH)

In [16]:
def clean(txt):
    txt = html.unescape(str(txt))
    txt = emoji.replace_emoji(txt, replace='')
    txt = re.sub(r"http\S+|www\.\S+", " ", txt)
    txt = re.sub(r"@\w+", " ", txt)
    txt = re.sub(r"#", "", txt)
    txt = re.sub(r"\s{2,}", " ", txt)
    return txt.strip()

In [17]:
df["clean_text"] = df["text"].astype(str).map(clean)
df = df.dropna(subset=["clean_text","label"])
df = df[df["clean_text"].str.len() > 0]

# chronological split (70 / 15 / 15)
df = df.sort_values("timestamp")
n  = len(df)
train_df = df.iloc[: int(0.70*n)]
val_df   = df.iloc[int(0.70*n): int(0.85*n)]
test_df  = df.iloc[int(0.85*n):]

tweet_ds = DatasetDict({
    "train":      Dataset.from_pandas(train_df[["clean_text","label"]]),
    "validation": Dataset.from_pandas(val_df[["clean_text","label"]]),
    "test":       Dataset.from_pandas(test_df[["clean_text","label"]]),
})
tweet_ds = tweet_ds.cast_column("label", ClassLabel(num_classes=2))
tweet_ds

Casting the dataset:   0%|          | 0/14439 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3094 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3095 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['clean_text', 'label', '__index_level_0__'],
        num_rows: 14439
    })
    validation: Dataset({
        features: ['clean_text', 'label', '__index_level_0__'],
        num_rows: 3094
    })
    test: Dataset({
        features: ['clean_text', 'label', '__index_level_0__'],
        num_rows: 3095
    })
})

In [18]:
from transformers import AutoTokenizer
MODEL_CKPT = "distilbert-base-uncased"     # ⚖️  much smaller than BERT
tok = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize(batch):
    return tok(batch["clean_text"],
               padding="max_length",
               truncation=True,
               max_length=128)

tweet_ds = tweet_ds.map(tokenize, batched=True, remove_columns=["clean_text"])
tweet_ds.set_format("torch",
                    columns=["input_ids","attention_mask","label"])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/14439 [00:00<?, ? examples/s]

Map:   0%|          | 0/3094 [00:00<?, ? examples/s]

Map:   0%|          | 0/3095 [00:00<?, ? examples/s]

In [20]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

MODEL_CKPT = "distilbert-base-uncased"

# 1️⃣  Load the base classifier head with 2 labels
base = AutoModelForSequenceClassification.from_pretrained(
           MODEL_CKPT, num_labels=2)

# 2️⃣  Tell PEFT which modules to LoRA‑tize
DISTIL_TARGET = [
    "q_lin",      # query
    "k_lin",      # key
    "v_lin",      # value
    "out_lin",    # attn output
    "ffn.lin1",   # first FFN linear
    "ffn.lin2",   # second FFN linear
]

lora_cfg = LoraConfig(
    task_type      = TaskType.SEQ_CLS,
    r              = 8,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    target_modules = DISTIL_TARGET      # 👈  mandatory for DistilBERT
)

# 3️⃣  Wrap with LoRA
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,255,682 || all params: 68,210,692 || trainable%: 1.8409


In [22]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), dim=-1)
    acc   = accuracy_score(labels, preds)
    p,r,f,_ = precision_recall_fscore_support(
                  labels, preds, average="macro", zero_division=0)
    return {"accuracy":acc, "precision":p, "recall":r, "f1":f}

args = TrainingArguments(
    output_dir               = "/content/tweet_cls_lora",
    num_train_epochs         = 3,
    per_device_train_batch_size = 8,     # tiny VRAM footprint
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 4,     # → effective batch 32
    learning_rate            = 2e-4,
    weight_decay             = 0.01,
    eval_strategy      = "epoch",
    save_strategy            = "epoch",
    load_best_model_at_end   = True,
    metric_for_best_model    = "f1",
    greater_is_better        = True,
    fp16                     = torch.cuda.is_available(),
    seed                     = 42,
    logging_steps            = 100,
)

trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = tweet_ds["train"],
    eval_dataset    = tweet_ds["validation"],
    tokenizer       = tok,
    compute_metrics = metrics_fn,
)

trainer.train()
print("Test‑set metrics:", trainer.evaluate(tweet_ds["test"]))

<ipython-input-22-73f18a3e9fe1>:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.280300,0.244414,0.926955,0.500000,0.463478,0.481047
2,0.249000,0.233867,0.948610,0.500000,0.474305,0.486814


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.280300,0.244414,0.926955,0.500000,0.463478,0.481047
2,0.209800,0.302963,0.930511,0.500000,0.465255,0.482002


Test‑set metrics: {'eval_loss': 0.29262489080429077, 'eval_accuracy': 0.9195476575121163, 'eval_precision': 0.5, 'eval_recall': 0.45977382875605816, 'eval_f1': 0.4790439319979801, 'eval_runtime': 780.5611, 'eval_samples_per_second': 3.965, 'eval_steps_per_second': 0.496, 'epoch': 2.9950138504155124}


In [23]:
ADAPTER_DIR = "/content/tweet_cls_lora_distilbert"
model.save_pretrained(ADAPTER_DIR)